# IMD station collector (Colab)

**[CEEW Scraping Field Course](https://mrafiuddin50.github.io/ceew_scraping_curriculum/), Module 06 (Route B) and Track B, lesson B1.**

This notebook collects today's observations, the 7-day forecast and the 14-day temperature table for every IMD station in one state, from [IMD's Local Weather Report & Forecast](https://city.imd.gov.in/citywx/responsive/). It can also collect each station's climate page (15-year October extremes and 1991–2020 normals). Results go to a Google Sheet or an Excel file in your Google Drive.

Everything runs on Google's servers. Nothing is installed on your computer.

**How to use it:** run the steps in order with the ▶ button beside each one. Change settings only in Step 1.

**Rules this notebook follows, and you should too:**
- At least 1 second between requests (2 seconds by default). Collect one state, not all of India.
- IMD's placeholders (`NA`, `99.9`, `999`, `()`, `0000-00-00`) become blanks. Nothing is filled in or rounded.
- Stations that return no data are listed on their own tab, not silently dropped.
- The raw replies are kept on a separate tab, so every number can be traced back.

**Status:** tested on 12 October 2026 against saved IMD replies (Delhi and the Safdarjung climate page), not yet from Colab against the live site. Step 3 checks the connection first. If it fails, use the Claude route in Module 06 and tell the course owner.

In [ ]:
#@title Step 1: Settings { display-mode: "form" }
STATE = "Delhi" #@param ["Andaman & Nicobar", "Andhra Pradesh", "Arunachal Pradesh", "Assam", "Bihar", "Chhatisgarh", "Delhi", "Diu", "Goa", "Gujarat", "Haryana", "Himachal Pradesh", "Jammu & Kashmir", "Jharkhand", "Karnataka", "Kerala", "Ladakh UT", "Lakshadweep", "Madhya Pradesh", "Maharashtra", "Manipur", "Meghalaya", "Mizoram", "Nagaland", "Orissa", "Pondicherry", "Punjab", "Rajasthan", "Sikkim", "Tamil Nadu", "Telangana", "Tripura", "Uttar Pradesh", "Uttrakhand", "West Bengal"]
#@markdown Leave blank to collect every station in the state, or list station IDs separated by commas (e.g. `42182, 42181`).
STATION_IDS = "" #@param {type:"string"}
#@markdown Seconds between requests. Don't go below 2 unless the course owner agrees.
PAUSE_SECONDS = 2 #@param {type:"slider", min:1, max:5, step:0.5}
#@markdown Also collect each station's climate page (doubles the number of requests).
INCLUDE_CLIMATE_PAGES = False #@param {type:"boolean"}
SAVE_TO = "Google Sheet" #@param ["Google Sheet", "Excel file in Google Drive", "Both"]
print("Settings saved. Run Step 2.")

In [ ]:
#@title Step 2: Load the working parts (read the comments if you're curious) { display-mode: "form" }
# --- IMD collector: the working parts ------------------------------------
# You don't need to edit anything in this cell. Read the comments to see
# what each part does; Module 06 of the course explains the ideas.

import json, re, time, datetime as dt
from io import StringIO
import requests
import pandas as pd

BASE = "https://city.imd.gov.in/citywx/responsive/api/"
CLIMATE = "https://city.imd.gov.in/citywx/extreme/climate.php?id="
IST = dt.timezone(dt.timedelta(hours=5, minutes=30))

# Say who we are. Being open about the purpose is part of polite collection.
HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; CEEW-research-collector/2.0; non-commercial research)",
    "Referer": "https://city.imd.gov.in/citywx/responsive/",
    "Accept": "application/json, text/html;q=0.9, */*;q=0.8",
}

# ---- 1. Turning IMD's placeholders into blanks ---------------------------
# IMD writes "no data" in several ways: NA, empty text, (), 0000-00-00, and
# impossible numbers such as 99.9 degrees or 999 % humidity. We check each
# field against what is physically possible rather than a fixed list, so a
# real 99.0 mm of rain is kept but a 99.9 degree temperature is not.
TEXT_PLACEHOLDERS = {"", "NA", "N/A", "-", "--", "()", "0000-00-00", "0000-00-00 00:00:00"}
LIMITS = {                      # (lowest plausible, highest plausible)
    "temperature": (-60.0, 60.0),
    "departure": (-30.0, 30.0),
    "humidity": (0.0, 100.0),
    "rainfall": (0.0, 1500.0),
}

def as_text(value):
    """Plain text, or None if IMD sent a placeholder."""
    if value is None:
        return None
    s = str(value).strip()
    return None if s in TEXT_PLACEHOLDERS else s

def as_number(value, kind):
    """A number within physical limits for its kind, or None."""
    s = as_text(value)
    if s is None:
        return None
    try:
        x = float(s)
    except ValueError:
        return None
    low, high = LIMITS[kind]
    return x if low <= x <= high else None

# ---- 2. Talking to IMD politely --------------------------------------------
class Collector:
    def __init__(self, pause_seconds=2.0, max_failures_in_a_row=5):
        self.pause = max(1.0, float(pause_seconds))   # never faster than 1 request a second
        self.max_fail = max_failures_in_a_row
        self.session = requests.Session()             # keeps IMD's session cookie between requests
        self.session.headers.update(HEADERS)
        self.fail_streak = 0

    def _wait(self):
        time.sleep(self.pause)

    def station_list(self):
        """All states and their stations, as IMD's own menu lists them."""
        r = self.session.get(BASE, timeout=30)
        r.raise_for_status()
        data = r.json().get("data") or {}
        return {state: {sid: name.strip() for sid, name in stations.items()} for state, stations in data.items()}

    def station(self, station_id):
        """One station's record. Returns (data, problem). Exactly one is None."""
        for attempt in (1, 2):
            try:
                r = self.session.post(BASE + "fetchCity_static.php", data={"ID": str(station_id)}, timeout=30)
                if r.status_code == 429:
                    return None, "IMD said too many requests (429). Stop and slow down."
                r.raise_for_status()
                data = r.json()
            except requests.exceptions.SSLError:
                return None, "Security certificate check failed. Do not switch checks off; tell the course owner."
            except Exception as e:
                if attempt == 1:
                    time.sleep(self.pause * 3)
                    continue
                return None, f"Request failed: {type(e).__name__}"
            if isinstance(data, list) and data and isinstance(data[0], dict):
                if str(data[0].get("station_id")) != str(station_id):
                    return None, f"Reply was for station {data[0].get('station_id')}, not {station_id}"
                return data, None
            if isinstance(data, dict) and data.get("error"):
                return None, f"IMD replied: {data.get('error')} (status {data.get('status')})"
            return None, "Unexpected reply format"
        return None, "Request failed"

    def climate_tables(self, url):
        """The two tables on a station's climate page, or (None, None)."""
        if not url:
            return None, None
        r = self.session.get(url, timeout=30)
        r.raise_for_status()
        return parse_climate_html(r.text)

# ---- 3. Reading a station record into tidy rows ----------------------------
def parse_station(data, state):
    rec = data[0]
    sid, name = str(rec.get("station_id")), as_text(rec.get("station"))
    page_date = as_text(rec.get("dat"))
    obs = {
        "state": state, "station_id": sid, "station": name,
        "lat": _coord(rec.get("lat"), -90, 90),
        "lon": _coord(rec.get("lon"), -180, 180),
        "page_date": page_date,
        "max_temp_c": as_number(rec.get("max"), "temperature"),
        "max_temp_date": as_text(rec.get("max_date")),          # the day the maximum belongs to
        "max_departure_c": as_number(rec.get("maxdep"), "departure"),
        "min_temp_c": as_number(rec.get("min"), "temperature"),
        "min_departure_c": as_number(rec.get("mindep"), "departure"),
        "rainfall_mm": as_number(rec.get("rainfall"), "rainfall"),
        "rh_0830_pct": as_number(rec.get("rh0830"), "humidity"),
        "rh_1730_pct": as_number(rec.get("rh1730"), "humidity"),
        "sunrise": as_text(rec.get("sunrise")), "sunset": as_text(rec.get("sunset")),
        "imd_updated": as_text(rec.get("updat")),
        "warning_today": as_text((rec.get("wText") or [None])[0]),
        "warning_colour_today": as_text((rec.get("wColor") or [None])[0]),
        "climate_page": as_text(rec.get("extreme_url")),
    }
    forecast = []
    base = dt.date.fromisoformat(page_date) if page_date else None
    lists = {k: rec.get(k) or [] for k in ("ffc", "fmax", "fmin", "rh0830d", "rh1730d", "wColor", "wText")}
    for i in range(7):
        pick = lambda k: lists[k][i] if i < len(lists[k]) else None
        forecast.append({
            "state": state, "station_id": sid, "station": name, "day": i,
            "forecast_date": (base + dt.timedelta(days=i)).isoformat() if base else None,
            "description": as_text(pick("ffc")),
            "max_temp_c": as_number(pick("fmax"), "temperature"),
            "min_temp_c": as_number(pick("fmin"), "temperature"),
            "rh_0830_pct": as_number(pick("rh0830d"), "humidity"),
            "rh_1730_pct": as_number(pick("rh1730d"), "humidity"),
            "warning_colour": as_text(pick("wColor")),
            "warning": as_text(pick("wText")),
        })
    trends = []
    if len(data) > 1 and isinstance(data[1], dict):
        for day, v in sorted(data[1].items()):
            if isinstance(v, dict):
                trends.append({
                    "state": state, "station_id": sid, "station": name, "date": day,
                    "observed_max_c": as_number(v.get("MAX"), "temperature"),
                    "observed_min_c": as_number(v.get("MIN"), "temperature"),
                    "normal_max_c": as_number(v.get("MAX_N"), "temperature"),
                    "normal_min_c": as_number(v.get("MIN_N"), "temperature"),
                    "forecast_max_c": as_number(v.get("F_MAX"), "temperature"),
                    "forecast_min_c": as_number(v.get("F_MIN"), "temperature"),
                })
    return obs, forecast, trends

def _coord(value, low, high):
    s = as_text(value)
    try:
        x = float(s)
    except (TypeError, ValueError):
        return None
    return x if low <= x <= high and x != 0 else None

# ---- 4. Reading the climate page --------------------------------------------
VALUE_DAY = re.compile(r"^\s*(-?\d+(?:\.\d+)?)?\s*\(([^)]*)\)\s*$")

def split_value_day(cell):
    """'36.7 (10)' -> (36.7, '10'); ' ()' -> (None, None); '5.4' -> (5.4, None)."""
    s = as_text(cell)
    if s is None:
        return None, None
    m = VALUE_DAY.match(s)
    if m:
        return (float(m.group(1)) if m.group(1) else None), (m.group(2) or None)
    try:
        return float(s), None
    except ValueError:
        return None, None

def parse_climate_html(html):
    tables = pd.read_html(StringIO(html), header=None)
    if len(tables) < 2:
        return None, None
    ext_raw, norm_raw = tables[0], tables[1]
    # Keep only data rows, recognised by their first cell (a year or "All Time";
    # a month or "Annual"). Don't count header rows: pandas may already have
    # turned them into column names, and skipping a fixed number drops real data.
    extremes = []
    for row in ext_raw.itertuples(index=False):
        cells = ["" if pd.isna(c) else str(c) for c in row]
        if not re.fullmatch(r"\d{4}|All Time", cells[0].strip()):
            continue
        hmax, hmax_day = split_value_day(cells[1])
        lmin, lmin_day = split_value_day(cells[2])
        rain, rain_day = split_value_day(cells[3])
        total, total_when = split_value_day(cells[4])
        extremes.append({"period": cells[0], "highest_max_c": hmax, "highest_max_day": hmax_day,
                         "lowest_min_c": lmin, "lowest_min_day": lmin_day,
                         "max_24h_rain_mm": rain, "max_24h_rain_day": rain_day,
                         "monthly_total_mm": total, "monthly_total_when": total_when})
    names = ["month", "mean_min_temp_c", "mean_max_temp_c", "mean_rainfall_mm", "rainy_days",
             "days_hail", "days_thunder", "days_fog", "days_squall"]
    normals = []
    months = {"Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec","Annual"}
    for row in norm_raw.itertuples(index=False):
        cells = list(row)[:len(names)]
        if str(cells[0]).strip() not in months:
            continue
        rec = {"month": str(cells[0])}
        for k, c in zip(names[1:], cells[1:]):
            try:
                rec[k] = float(c)
            except (TypeError, ValueError):
                rec[k] = None
        normals.append(rec)
    return pd.DataFrame(normals), pd.DataFrame(extremes)

# ---- 5. A quick health check on what we collected ---------------------------
def summary_checks(obs_df, failures_df, expected):
    lines = [f"Stations IMD lists for this selection: {expected}",
             f"Stations with a record: {len(obs_df)}",
             f"Stations with no record: {len(failures_df)}"]
    if len(obs_df) + len(failures_df) != expected:
        lines.append("WARNING: records plus failures don't add up to the station count. Investigate before using the data.")
    if len(obs_df):
        blank_max = int(obs_df["max_temp_c"].isna().sum())
        lines.append(f"Stations with no maximum temperature: {blank_max}")
        dup = obs_df["station_id"].duplicated().sum()
        if dup:
            lines.append(f"WARNING: {dup} duplicate station IDs")
        dates = obs_df["max_temp_date"].dropna().unique().tolist()
        lines.append(f"Maximum temperatures are for: {', '.join(sorted(dates)) or 'no date given'} "
                     f"(page date: {', '.join(sorted(obs_df['page_date'].dropna().unique()))})")
    return "\n".join(lines)

print('Ready. Run Step 3.')

In [ ]:
#@title Step 3: Check the connection and find the stations { display-mode: "form" }
col = Collector(pause_seconds=PAUSE_SECONDS)
stations = {}
checked_state = None
try:
    directory = col.station_list()
except Exception as e:
    directory = {}
    print(f"Could not reach IMD from Colab: {type(e).__name__}: {e}")
    print("Use the Claude route in Module 06 instead, and tell the course owner what this message said.")

if directory:
    total = sum(len(v) for v in directory.values())
    print(f"Connected. IMD lists {total} stations in {len(directory)} states and UTs.")
    if STATE not in directory:
        print(f"'{STATE}' is not in IMD's list. IMD's own spellings are:")
        print(", ".join(sorted(directory)))
    else:
        stations = directory[STATE]
        checked_state = STATE
        wanted = [s.strip() for s in STATION_IDS.split(",") if s.strip()]
        if wanted:
            missing = [s for s in wanted if s not in stations]
            if missing:
                print(f"Not found in {STATE}: {', '.join(missing)}")
            stations = {s: stations[s] for s in wanted if s in stations}
        requests_needed = len(stations) * (2 if INCLUDE_CLIMATE_PAGES else 1)
        minutes = requests_needed * PAUSE_SECONDS / 60
        print(f"{STATE}: {len(stations)} stations selected. About {requests_needed} requests, roughly {minutes:.0f} minute(s).")
        print("Run Step 4 to collect.")

In [ ]:
#@title Step 4: Collect { display-mode: "form" }
if not stations or checked_state != STATE:
    raise SystemExit("No stations selected for this state. Run Step 3 again (after changing Step 1, always rerun Step 3).")
started = dt.datetime.now(IST)
climate_problems = []
obs_rows, fc_rows, trend_rows, normal_rows, extreme_rows, failures, raw = [], [], [], [], [], [], []
fail_streak = 0
for n, (sid, name) in enumerate(stations.items(), 1):
    print(f"[{n}/{len(stations)}] {name} ({sid})", end=" ... ")
    data, problem = col.station(sid)
    if data:
        fail_streak = 0
        raw.append({"station_id": sid, "reply_json": json.dumps(data, ensure_ascii=False)})
        try:
            obs, fc, tr = parse_station(data, STATE)
        except Exception as e:
            failures.append({"station_id": sid, "station": name, "problem": f"Reply couldn't be read: {type(e).__name__}"})
            print("reply couldn't be read")
            col._wait()
            continue
        obs_rows.append(obs); fc_rows += fc; trend_rows += tr
        print("ok")
        if INCLUDE_CLIMATE_PAGES and obs["climate_page"]:
            col._wait()
            try:
                normals, extremes = col.climate_tables(obs["climate_page"])
                if normals is not None:
                    for r in normals.to_dict("records"):
                        normal_rows.append({"station_id": sid, "station": name, **r})
                    for r in extremes.to_dict("records"):
                        extreme_rows.append({"station_id": sid, "station": name, **r})
            except Exception as e:
                climate_problems.append({"station_id": sid, "station": name, "problem": f"Climate page: {type(e).__name__}"})
    else:
        fail_streak += 1
        failures.append({"station_id": sid, "station": name, "problem": problem})
        print(problem)
        if "429" in problem or fail_streak >= col.max_fail:
            print("Stopping early: too many failures in a row. Wait an hour before trying again.")
            break
    col._wait()

obs_df, fc_df, trend_df = pd.DataFrame(obs_rows), pd.DataFrame(fc_rows), pd.DataFrame(trend_rows)
fail_df = pd.DataFrame(failures, columns=["station_id", "station", "problem"])
normals_df, extremes_df = pd.DataFrame(normal_rows), pd.DataFrame(extreme_rows)
raw_df = pd.DataFrame(raw)
climate_problems_df = pd.DataFrame(climate_problems, columns=["station_id", "station", "problem"])
print(f"\nFinished in {(dt.datetime.now(IST) - started).seconds // 60} min. Run Step 5.")

In [ ]:
#@title Step 5: Check before you use it { display-mode: "form" }
print(summary_checks(obs_df, fail_df, len(stations)))
if INCLUDE_CLIMATE_PAGES:
    print(f"Climate pages collected: {normals_df['station_id'].nunique() if len(normals_df) else 0}; problems: {len(climate_problems_df)}")
if len(fail_df):
    print("\nNo record:")
    print(fail_df.to_string(index=False))
print("\nFirst rows of the observations:")
print(obs_df.head(10).to_string(index=False) if len(obs_df) else "(none)")
print("\nNow do the five checks in Module 07, including five spot-checks against the live site.")

In [ ]:
#@title Step 6: Save to Google Sheets or Drive { display-mode: "form" }
stamp = started.strftime("%Y-%m-%d %H%M IST")
title = f"IMD {STATE} {stamp}"
about = pd.DataFrame([
    ["Source", "IMD Local Weather Report & Forecast, https://city.imd.gov.in/citywx/responsive/"],
    ["Data feed", "GET .../api/ (station list); POST .../api/fetchCity_static.php with form field ID (station data)"],
    ["Climate pages", "https://city.imd.gov.in/citywx/extreme/climate.php?id=<station_id>" if INCLUDE_CLIMATE_PAGES else "Not collected"],
    ["Collected (IST)", started.strftime("%Y-%m-%d %H:%M")],
    ["State", STATE],
    ["Stations selected", str(len(stations))],
    ["Records / no record", f"{len(obs_df)} / {len(fail_df)}"],
    ["Pause between requests (s)", str(PAUSE_SECONDS)],
    ["Placeholders blanked", "NA, empty, (), 0000-00-00, and values outside physical limits (temperature -60 to 60 C, departure -30 to 30, humidity 0-100, rainfall 0-1500 mm)"],
    ["max_temp_c", "Belongs to max_temp_date, usually the day before page_date"],
    ["Caveat", "Not official IMD records. Spot-check against the live site before use in any CEEW output."],
], columns=["item", "detail"])
tabs = {"about": about, "observations": obs_df, "forecast_7day": fc_df, "temperature_14day": trend_df,
        "no_record": fail_df, "raw_replies": raw_df}
if INCLUDE_CLIMATE_PAGES:
    tabs["climate_normals"] = normals_df
    tabs["climate_extremes"] = extremes_df
    tabs["climate_problems"] = climate_problems_df

def to_values(df):
    clean = df.astype(object).where(pd.notna(df), "")
    return [list(map(str, df.columns))] + clean.values.tolist()

if SAVE_TO in ("Google Sheet", "Both"):
    from google.colab import auth
    auth.authenticate_user()
    import gspread
    from google.auth import default
    creds, _ = default()
    gc = gspread.authorize(creds)
    sh = gc.create(title)
    first = True
    for name, df in tabs.items():
        values = to_values(df) if len(df.columns) else [["(empty)"]]
        if first:
            ws = sh.sheet1; ws.update_title(name); first = False
        else:
            ws = sh.add_worksheet(title=name, rows=max(len(values), 1) + 5, cols=max(len(values[0]), 1) + 2)
        ws.update(values=values, range_name="A1")
    print(f"Google Sheet created: {sh.url}")

if SAVE_TO in ("Excel file in Google Drive", "Both"):
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    folder = "/content/drive/MyDrive/IMD_collections"
    os.makedirs(folder, exist_ok=True)
    path = f"{folder}/{title}.xlsx"
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        for name, df in tabs.items():
            df.to_excel(xw, sheet_name=name[:31], index=False)
    print(f"Saved to Google Drive: My Drive/IMD_collections/{title}.xlsx")

## Next steps

1. **Validate** (Module 07): row count, placeholders, ranges, duplicate IDs, five spot-checks against the live site.
2. **Log** (Module 08): note the date, the settings and anything odd. The `about` tab is a start, not the whole log.
3. **Don't run this on a loop.** For daily collection, use the Apps Script in Track B, lesson B3, which runs once a day at a fixed time.